# 02 Build Qdrant Index

Generate fine-tuned SigLIP2 and BGE-M3 embeddings, then upsert segment points to Qdrant Cloud.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -r /content/drive/MyDrive/cooking_mvp_project/requirements-colab.txt

In [ ]:
import os, sys
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path('/content/drive/MyDrive/cooking_mvp_project')
sys.path.insert(0, str(PROJECT_ROOT))

os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
SEGMENTS = PROJECT_ROOT / 'data' / 'processed' / 'canonical_segments.parquet'
ADAPTER_PATH = '/content/drive/MyDrive/korean_cooking_shorts_dataset/siglip2_lora_qv_r16_best'

In [ ]:
import pandas as pd
from src.index.qdrant_client import get_qdrant_client
from src.index.build_index import build_and_upsert
from src.models.bge_encoder import BGEEncoder
from src.models.siglip_encoder import SigLIPEncoder

segments = pd.read_parquet(SEGMENTS)
client = get_qdrant_client()
siglip = SigLIPEncoder(adapter_path=ADAPTER_PATH)
bge = BGEEncoder()
build_and_upsert(segments, client, siglip, bge, recreate=True, batch_size=32)